En nuestra propuesta de proyecto integrador mencionamos el uso de Apache Kafka o AWS Kinesis para la ingesta masiva de eventos en tiempo real (modelo productor-suscriptor) , acoplado a un procesamiento Spark Structured Streaming. Sin embargo, en la práctica (y por limitaciones de la versión gratuita de Databricks), vamos a realizar en este paso una ingesta Batch descargando los archivos estáticos desde Kaggle hacia un Volume.
*Nuestra estrategia híbrida (Batch simulando Streaming):*
Para mantener la coherencia con la propuesta de "Arquitectura de Datos de Alto Rendimiento", diseñaremos el pipeline de tal manera que, aunque los datos base sean estáticos, el código esté preparado para un entorno de streaming.
- Utilizaremos la funcionalidad Auto Loader (cloudFiles) de Databricks (si está disponible en tu entorno) o lecturas incrementales mediante readStream procesando los CSV estáticos como si fueran un flujo continuo de datos, depositándolos en nuestra capa Bronze.

Diseño de la Arquitectura Medallion (Bronze, Silver, Gold):
- El objetivo de esta arquitectura es refinar iterativamente los datos, garantizando calidad, linaje y rendimiento. Como estándar en Databricks, no usaremos simplemente archivos Parquet, sino Delta Tables (que están basadas en Parquet pero añaden la capa transaccional ACID)

El siguiente código lee el archivo CSV que descargamos desde Kaggle y fue almacenado en la carpeta e-commerce

In [0]:
#programa para leer los archivos en la carpeta e-commerce

import os

ruta = '/Volumes/workspace/default/ecommerce_raw'

archivos = os.listdir(ruta)

print("="*31)
print("LECTURA DE ARCHIVOS ALMACENADOS\nEN CARPETA ecommerce_raw")
print("="*31)


for f in archivos:
    size = os.path.getsize(f'{ruta}/{f}')
    print(f"📄 {f}  →  {size/1e9:.2f} GB")

print()

# Capa Bronze
Objetivo: Almacenar los datos tal cual llegaron (inmutabilidad), sirviendo como fuente de la verdad para reprocesamientos.
- Qué hacemos aquí?: Leer los CSV del volumen /Volumes/workspace/default/ecommerce_raw y escribirlos en formato Delta sin alterar esquemas ni limpiar datos.

- Por qué?: Si en el futuro descubrimos que limpiamos mal una variable en la capa Silver, siempre podemos volver a la Bronze sin tener que re-descargar de Kaggle.

In [0]:
# 01_Ingestion_Bronze.ipynb

from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.functions import current_timestamp, col

# 1. Crear el Volumen de Unity Catalog programáticamente
print("1. Verificando/Creando Volumen 'e_commerce'...")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.e_commerce")

# 2. Definición estricta de esquema
schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("user_id", StringType(), True),
    StructField("user_session", StringType(), True)
])

# 3. Definición de rutas según tu nueva arquitectura
# Leemos de donde descargaste originalmente, escribimos en la nueva estructura
raw_data_path = "/Volumes/workspace/default/ecommerce_raw/*.csv"
bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"

# 4. Lectura de los CSVs optimizada
print("2. Leyendo datos Raw...")
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(raw_data_path)

# 5. Añadir metadatos de linaje (Unity Catalog)
print("3. Añadiendo metadatos de linaje...")
df_bronze = df_raw \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

# # 6. Escritura en formato Delta (Capa Bronze)
# # Databricks creará automáticamente la subcarpeta 'bronze/clickstream' dentro del Volumen 'e_commerce'
# print("4. Escribiendo en formato Delta (Capa Bronze)... Esto tomará unos minutos.")
# (df_bronze.write
#     .format("delta")
#     .mode("append") 
#     .save(bronze_table_path))

# print("✅ Pipeline ejecutado con éxito. Datos listos en capa Bronze.")


# 6. Escritura en formato Delta (Capa Bronze BLINDADA)
print("Escribiendo en formato Delta (Capa Bronze)...")
(df_bronze.write
    .format("delta")
    .mode("overwrite") # <--- Este es el punto clave para la idempotencia en Batch
    .option("overwriteSchema", "true") # Opcional: útil si cambias el esquema en el futuro
    .save(bronze_table_path))

print("✅ Pipeline ejecutado con éxito. Datos listos en capa Bronze\nOperación idempotente garantizada.")

# Silver

1. Tipado de Datos (Casting)

El código toma columnas que llegaron como texto plano y las convierte a sus tipos matemáticos correctos. La fecha de evento se convierte a un formato de tiempo continuo (t), y el precio pasa a ser una variable numérica continua (FloatType). Esto es vital, ya que los algoritmos matemáticos no pueden calcular recencias ni promedios sobre cadenas de texto.

2. Feature Engineering Temporal:

A partir de la marca de tiempo original (t), el motor distribuido extrajo características cíclicas y discretas:

- La fecha exacta (date).

- La hora del día (0,23).

- El día de la semana (1,7) y su nombre.

Esto permite que, en el EDA y modelado posterior, podamos encontrar patrones de propensión basados en ventanas horarias (por ejemplo, si la franja matutina tiene mayor intención de compra).

3. Aplanamiento Taxonómico Seguro (Parsing):

La columna category_code venía anidada (ej. electronics.smartphone.apple). El código utiliza una división por arreglos (arrays) e implementa una validación lógica con size() para extraer de manera segura la macro-categoría, sub-categoría y tipo de ítem. Si a un evento le faltan niveles taxonómicos, el código lo maneja sin colapsar, asignando un valor nulo de forma controlada.

4. Aseguramiento de Calidad (Data Quality & Imputation)

Aplicamos el principio de que la ausencia de información también es información.

* Imputación: Llenamos los valores faltantes en categorías y marcas con "Unknown", preservando el evento en lugar de borrarlo, lo cual es fundamental para los árboles de decisión.

* Limpieza de Ruido: Eliminamos (dropna) cualquier evento que no tuviera user_session o user_id. Un log sin sesión es un evento "huérfano" que matemáticamente no puede unirse a un customer journey, por lo que carece de valor predictivo.

5. Escritura Idempotente:

Finalmente, el script utiliza .mode("overwrite"). Esto garantiza que, sin importar cuántas veces ejecuten el notebook, la tabla resultante siempre representará el estado correcto y limpio de los datos, eliminando el riesgo de duplicidad masiva.

In [0]:
from pyspark.sql.functions import col, split, to_timestamp, to_date, hour, dayofweek, date_format, size, when, lit

bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"
silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"

df_bronze = spark.read.format("delta").load(bronze_table_path)

# Ampliación de Feature Engineering en Silver (Blindado contra arrays irregulares)
df_silver = df_bronze \
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'")) \
    .withColumn("price", col("price").cast("float")) \
    .withColumn("date", to_date(col("event_time"))) \
    .withColumn("hour", hour(col("event_time"))) \
    .withColumn("day_of_week_num", dayofweek(col("event_time"))) \
    .withColumn("day_name", date_format(col("event_time"), "EEEE")) \
    .withColumn("cat_array", split(col("category_code"), r"\.")) \
    .withColumn("macro_category", col("cat_array").getItem(0)) \
    .withColumn("sub_category", when(size(col("cat_array")) > 1, col("cat_array").getItem(1)).otherwise(lit("Unknown"))) \
    .withColumn("item_type", when(size(col("cat_array")) > 2, col("cat_array").getItem(2)).otherwise(lit("Unknown"))) \
    .drop("cat_array", "category_code") \
    .fillna({
        "brand": "Unknown",
        "macro_category": "Unknown"
    }) \
    .dropna(subset=["user_session", "user_id"])

# Guardar en Silver
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(silver_table_path))

print("✅ Capa Silver enriquecida, blindada y consolidada.")

Validación rápida de la capa Silver

In [0]:
# Lectura rápida de la capa Silver para validación
silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
df_silver_qa = spark.read.format("delta").load(silver_table_path)

# Seleccionamos las columnas nuevas que creamos y mostramos 10 registros donde sí había categoría
print("🔍 Validando Feature Engineering Temporal y de Taxonomía:")
df_silver_qa.select(
    "event_time", "date", "hour", "day_name", 
    "macro_category", "sub_category", "item_type"
).filter(col("macro_category") != "Unknown").show(10, truncate=False)

# Validar el esquema para asegurar los tipos de datos (ej. price como float)
df_silver_qa.printSchema()

In [0]:
from pyspark.sql.functions import max as spark_max, sum as spark_sum, when, col, round

# 1. Conectarnos a la capa Silver recién creada
silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
df_silver = spark.read.format("delta").load(silver_table_path)

# 2. Replicar la lógica del "Funnel por Unidad" (producto-en-sesión)
# Pivotamos los eventos para saber si en esa sesión se vio, se agregó al carrito o se compró ese producto.
df_funnel_unit = df_silver.groupBy("user_session", "product_id", "macro_category").agg(
    spark_max(when(col("event_type") == "view", 1).otherwise(0)).alias("has_view"),
    spark_max(when(col("event_type") == "cart", 1).otherwise(0)).alias("has_cart"),
    spark_max(when(col("event_type") == "purchase", 1).otherwise(0)).alias("has_purchase")
)

# 3. Calcular las métricas globales del Funnel
total_units = df_funnel_unit.count()

funnel_metrics = df_funnel_unit.agg(
    spark_sum("has_view").alias("total_views"),
    spark_sum("has_cart").alias("total_carts"),
    spark_sum("has_purchase").alias("total_purchases")
).collect()[0]

# Extraer resultados
views = funnel_metrics["total_views"]
carts = funnel_metrics["total_carts"]
purchases = funnel_metrics["total_purchases"]

# Fórmulas de conversión matemática
cart_rate = (carts / views) * 100 if views > 0 else 0
# Abandono = De los que llegaron al carrito, ¿cuántos NO compraron?
abandonment_rate = ((carts - purchases) / carts) * 100 if carts > 0 else 0
conversion_rate = (purchases / views) * 100 if views > 0 else 0

print(f"📊 Funnel Global (Sobre 14GB de datos):")
print(f"Unidades únicas evaluadas: {total_units:,}")
print(f"  - Llegaron a Vistas: {views:,}")
print(f"  - Llegaron a Carrito: {carts:,} (Tasa adición: {cart_rate:.2f}%)")
print(f"  - Compras efectivas: {purchases:,} (Tasa Conversión: {conversion_rate:.2f}%)")
print(f"  🚨 Tasa de Abandono de Carrito: {abandonment_rate:.2f}%")

Excelente, comnpas! Haber procesado 69 millones de unidades únicas sobre 14 GB de datos distribuidos no es poca cosa. Estmos viendo el poder real de Apache Spark en acción.


1. Interpretación de los Resultados del Funnel Masivo

- Tasa de Adición al Carrito (Cart Rate): En los 14 GB completos es de 3.85%. En la pequeña muestra del taller fue de 3.61%. Ambas cifras son consistentes.

- Tasa de Conversión (Conversion Rate): En Big Data es de 2.22%, mientras que en la muestra fue de 2.44%. Esto confirma la hipótesis original de nuestra propuesta de proyecto: la conversión oscila entre el 1% y el 3%, demostrando la ineficiencia estructural del e-commerce (el 97.78% no compra).

- Tasa de Abandono de Carrito (Abandonment Rate): ¡Aquí está el gran hallazgo! En los datos masivos, el abandono es del 42.27%. En la muestra del taller, habíamos calculado un 32.4%.

¿Por qué aumentó la tasa de abandono en casi 10 puntos porcentuales al procesar toda la data?
Matemáticamente, la varianza de una muestra pequeña puede no capturar toda la heterogeneidad de millones de usuarios. Al ver toda la base de datos (octubre y noviembre), estamos capturando picos de tráfico reales (como el Black Friday de noviembre) donde la gente añade cosas al carrito de forma impulsiva o para guardar precios, pero abandona mucho más que en un día promedio de octubre.

Esto confirma tu caso de negocio: ¡hay un segmento enorme de usuarios a un solo clic de distancia (tienen intención), pero que requieren un incentivo!